In [1]:
# Import required libraries
import requests
from bs4 import BeautifulSoup
import sqlite3
import pandas as pd
import re
from datetime import datetime

# Install any missing libraries (run in terminal if needed)
# pip install requests beautifulsoup4 pandas sqlite3

In [3]:
!pip install requests beautifulsoup4 psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 5.5 MB/s eta 0:00:00 MB/s eta 0:00:01:01


In [ ]:
import requests
from bs4 import BeautifulSoup
import psycopg2
from psycopg2 import sql, extras
import json
import re
import time
from urllib.parse import urljoin
import os
from dotenv import load_dotenv

# Load environment variables (optional)
load_dotenv()

# ============================================================
# POSTGRESQL CONFIGURATION
# ============================================================
# Replace these values with your connection details
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': int(os.getenv('DB_PORT', 5432)),
    'database': os.getenv('DB_NAME', 'films_db'),
    'user': os.getenv('DB_USER', 'postgres'),
    'password': os.getenv('DB_PASSWORD', 'postgres123')
}

# Or use environment variables:
# DB_CONFIG = {
#     'host': os.getenv('DB_HOST', 'localhost'),
#     'port': int(os.getenv('DB_PORT', 5432)),
#     'database': os.getenv('DB_NAME', 'films_db'),
#     'user': os.getenv('DB_USER', 'postgres'),
#     'password': os.getenv('DB_PASSWORD', '')
# }


def create_database_if_not_exists():
    """Creates the database if it doesn't exist"""
    # Connect to the default 'postgres' database
    conn = psycopg2.connect(
        host=DB_CONFIG['host'],
        port=DB_CONFIG['port'],
        user=DB_CONFIG['user'],
        password=DB_CONFIG['password'],
        database='postgres'
    )
    conn.autocommit = True
    cursor = conn.cursor()
    
    # Check if the database exists
    cursor.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB_CONFIG['database'],))
    exists = cursor.fetchone()
    
    if not exists:
        cursor.execute(sql.SQL("CREATE DATABASE {}").format(
            sql.Identifier(DB_CONFIG['database'])
        ))
        print(f"✅ Database '{DB_CONFIG['database']}' created")
    else:
        print(f"✅ Database '{DB_CONFIG['database']}' already exists")
    
    cursor.close()
    conn.close()


def get_db_connection():
    """Returns a connection to PostgreSQL"""
    return psycopg2.connect(
        host=DB_CONFIG['host'],
        port=DB_CONFIG['port'],
        database=DB_CONFIG['database'],
        user=DB_CONFIG['user'],
        password=DB_CONFIG['password']
    )


def extract_director_and_country_from_film_page(film_url):
    """
    Extracts director and country of origin from the film's Wikipedia page
    
    Args:
        film_url: URL of the film's Wikipedia page
        
    Returns:
        tuple: (director, country)
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        response = requests.get(film_url, headers=headers, timeout=10)
        if response.status_code != 200:
            return "Unknown", "Various"
        
        film_soup = BeautifulSoup(response.content, 'html.parser')
        
        # Look for infobox (Infobox film)
        infobox = film_soup.find('table', {'class': 'infobox'})
        if not infobox:
            infobox = film_soup.find('table', {'class': 'infobox vevent'})
        
        director = "Unknown"
        country = "Various"
        
        if infobox:
            # Search for director and country
            for row in infobox.find_all('tr'):
                header = row.find('th')
                if header:
                    header_text = header.get_text(strip=True).lower()
                    
                    # Look for director
                    if 'directed by' in header_text or 'director' in header_text:
                        value_cell = row.find('td')
                        if value_cell:
                            director_text = value_cell.get_text(separator=' ', strip=True)
                            director_text = re.sub(r'\[\d+\]', '', director_text)
                            director_text = re.sub(r'\([^)]*\)', '', director_text)
                            director = director_text.strip()
                            if director:
                                director = re.sub(r'\s+', ' ', director)
                    
                    # Look for country of origin
                    if 'country' in header_text or 'origin' in header_text or 'production' in header_text:
                        value_cell = row.find('td')
                        if value_cell:
                            country_text = value_cell.get_text(separator=' ', strip=True)
                            country_text = re.sub(r'\[\d+\]', '', country_text)
                            country_text = re.sub(r'\([^)]*\)', '', country_text)
                            # Take the first country if there are multiple
                            countries = [c.strip() for c in country_text.split('\n') if c.strip()]
                            if countries:
                                country = countries[0].split(',')[0].strip()
            
            # If director not found in infobox, search in text
            if director == "Unknown":
                content = film_soup.find('div', {'class': 'mw-parser-output'})
                if content:
                    first_paragraph = content.find('p')
                    if first_paragraph:
                        text = first_paragraph.get_text()
                        patterns = [
                            r'directed by\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)',
                            r'film was directed by\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)',
                            r'director\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)',
                        ]
                        for pattern in patterns:
                            match = re.search(pattern, text, re.IGNORECASE)
                            if match:
                                director = match.group(1).strip()
                                break
        
        # Clean up results
        director = re.sub(r'\s+', ' ', director).strip()
        country = re.sub(r'\s+', ' ', country).strip()
        director = re.sub(r'\[.*?\]', '', director)
        country = re.sub(r'\[.*?\]', '', country)
        
        return director, country
        
    except Exception as e:
        print(f"  Error extracting from {film_url}: {e}")
        return "Unknown", "Various"


def extract_films_data_with_follow_links():
    """
    Extracts film data from the main table and follows links
    to get director and country of origin
    """
    
    # Fetch Wikipedia page
    url = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Find the main table
    table = soup.find('table', {'class': 'wikitable'})
    if not table:
        print("Could not find the main table")
        return []
    
    films = []
    
    # Parse table rows
    rows = table.find_all('tr')[1:]  # Skip header row
    
    for idx, row in enumerate(rows, 1):
        cells = row.find_all(['td', 'th'])
        
        # Check if we have enough cells
        if len(cells) < 5:
            continue
        
        film = {}
        
        # Extract title and get link (column index 2 - Title)
        title_cell = cells[2]
        title_link = title_cell.find('a')
        
        if not title_link:
            continue
            
        film['title'] = title_link.get_text(strip=True)
        film['title'] = re.sub(r'\[[^\]]*\]', '', film['title']).strip()
        
        # Get the URL to the film's page
        film_url = urljoin("https://en.wikipedia.org/", title_link.get('href'))
        film['wiki_url'] = film_url
        
        # Extract box office (column index 3 - Worldwide gross)
        gross_cell = cells[3]
        gross_text = gross_cell.get_text(strip=True)
        gross_text = re.sub(r'\[[^\]]*\]', '', gross_text)  # Remove citations
        
        # Parse box office value
        gross_value = None
        try:
            # Remove any non-breaking spaces and normalize
            gross_text = gross_text.replace('\xa0', ' ')
            
            # Handle different formats
            if 'billion' in gross_text.lower():
                numbers = re.findall(r'[\d,]+(?:\.\d+)?', gross_text)
                if numbers:
                    value = float(numbers[0].replace(',', ''))
                    gross_value = value * 1000000000
            elif 'million' in gross_text.lower():
                numbers = re.findall(r'[\d,]+(?:\.\d+)?', gross_text)
                if numbers:
                    value = float(numbers[0].replace(',', ''))
                    gross_value = value * 1000000
            else:
                # Try to extract just numbers
                clean_text = re.sub(r'[^\d,\.]', '', gross_text)
                if clean_text:
                    gross_value = float(clean_text.replace(',', ''))
                    
        except Exception as e:
            print(f"  Error parsing box office '{gross_text}': {e}")
            gross_value = None
        
        film['box_office'] = gross_value
        
        # Extract year (column index 4 - Year)
        year_cell = cells[4]
        year_text = year_cell.get_text(strip=True)
        year_match = re.search(r'\d{4}', year_text)
        film['release_year'] = int(year_match.group()) if year_match else None
        
        # Only process if we have title and box office
        if film.get('title') and film.get('box_office'):
            print(f"\nProcessing {idx}. {film['title']} ({film['release_year']})...")
            print(f"  Box Office: ${film['box_office']:,.0f}")
            print(f"  URL: {film_url}")
            
            # Follow the link to get director and country
            director, country = extract_director_and_country_from_film_page(film_url)
            film['director'] = director
            film['country'] = country
            
            print(f"  Director: {director}")
            print(f"  Country: {country}")
            
            films.append(film)
            
            # Be polite - delay between requests
            time.sleep(0.5)
        else:
            if not film.get('title'):
                print(f"\nSkipping row {idx}: No title found")
            elif not film.get('box_office'):
                print(f"\nSkipping row {idx}: {film.get('title')} - No box office value found (text: '{gross_text}')")
    
    return films


def save_to_postgresql_and_json(films):
    """Saves data to PostgreSQL and exports to JSON"""
    
    # Create database if it doesn't exist
    create_database_if_not_exists()
    
    # Connect to the database
    conn = get_db_connection()
    cursor = conn.cursor()
    
    # Create table with extended data types
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS films (
            id SERIAL PRIMARY KEY,
            title TEXT NOT NULL,
            release_year INTEGER,
            director TEXT,
            box_office NUMERIC(15, 2),
            country TEXT,
            wiki_url TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    # Create indexes for query optimization
    cursor.execute('''
        CREATE INDEX IF NOT EXISTS idx_films_title ON films(title);
        CREATE INDEX IF NOT EXISTS idx_films_year ON films(release_year);
        CREATE INDEX IF NOT EXISTS idx_films_box_office ON films(box_office DESC);
        CREATE INDEX IF NOT EXISTS idx_films_country ON films(country);
    ''')
    
    # Clear existing data
    cursor.execute("TRUNCATE TABLE films RESTART IDENTITY CASCADE")
    
    # Insert data
    for film in films:
        cursor.execute('''
            INSERT INTO films (title, release_year, director, box_office, country, wiki_url)
            VALUES (%s, %s, %s, %s, %s, %s)
        ''', (
            film['title'],
            film['release_year'],
            film['director'],
            film['box_office'],
            film['country'],
            film.get('wiki_url', '')
        ))
    
    conn.commit()
    
    # Get data for JSON export
    cursor.execute("""
        SELECT id, title, release_year, director, box_office, country 
        FROM films 
        ORDER BY box_office DESC
    """)
    rows = cursor.fetchall()
    
    films_json = []
    for row in rows:
        films_json.append({
            'id': row[0],
            'title': row[1],
            'release_year': row[2],
            'director': row[3],
            'box_office': float(row[4]),
            'country': row[5]
        })
    
    # Save JSON file
    with open('films_data.json', 'w', encoding='utf-8') as f:
        json.dump(films_json, f, indent=2, ensure_ascii=False)
    
    # Get statistics
    cursor.execute("""
        SELECT 
            COUNT(*) as total_films,
            SUM(box_office) as total_revenue,
            AVG(box_office) as avg_revenue,
            MIN(release_year) as earliest_year,
            MAX(release_year) as latest_year,
            COUNT(DISTINCT country) as countries_count
        FROM films
    """)
    stats = cursor.fetchone()
    
    # Display summary
    print(f"\n{'='*60}")
    print(f"✅ SUCCESSFULLY COMPLETED")
    print(f"{'='*60}")
    print(f"PostgreSQL Database: {DB_CONFIG['database']}")
    print(f"Total films processed: {stats[0]}")
    print(f"Total revenue: ${stats[1]:,.0f}")
    print(f"Average revenue: ${stats[2]:,.0f}")
    print(f"Year range: {stats[3]} - {stats[4]}")
    print(f"Countries represented: {stats[5]}")
    print(f"JSON exported: films_data.json")
    
    # Top 10 films
    print(f"\n📊 TOP 10 FILMS:")
    for i, film in enumerate(films[:10], 1):
        print(f"{i}. {film['title']} ({film['release_year']})")
        print(f"   Director: {film['director']}")
        print(f"   Country: {film['country']}")
        print(f"   Box Office: ${film['box_office']:,.0f}")
    
    # Close connection
    cursor.close()
    conn.close()
    
    return films


def query_database_examples():
    """Example queries to the PostgreSQL database"""
    
    conn = get_db_connection()
    cursor = conn.cursor()
    
    print(f"\n{'='*60}")
    print("📊 DATABASE QUERY EXAMPLES")
    print(f"{'='*60}")
    
    # 1. Films by country
    print("\n1. Films by country:")
    cursor.execute("""
        SELECT country, COUNT(*) as film_count, SUM(box_office) as total_revenue
        FROM films
        GROUP BY country
        ORDER BY total_revenue DESC
        LIMIT 5
    """)
    for row in cursor.fetchall():
        print(f"   {row[0]}: {row[1]} films, ${row[2]:,.0f}")
    
    # 2. Top directors by box office
    print("\n2. Top directors by box office:")
    cursor.execute("""
        SELECT director, COUNT(*) as film_count, SUM(box_office) as total_revenue
        FROM films
        WHERE director != 'Unknown'
        GROUP BY director
        ORDER BY total_revenue DESC
        LIMIT 5
    """)
    for row in cursor.fetchall():
        print(f"   {row[0]}: {row[1]} films, ${row[2]:,.0f}")
    
    # 3. Films by decade
    print("\n3. Films by decade:")
    cursor.execute("""
        SELECT 
            (release_year / 10) * 10 as decade,
            COUNT(*) as film_count,
            SUM(box_office) as total_revenue
        FROM films
        GROUP BY decade
        ORDER BY decade DESC
    """)
    for row in cursor.fetchall():
        print(f"   {row[0]}s: {row[1]} films, ${row[2]:,.0f}")
    
    cursor.close()
    conn.close()


# ============================================================
# MAIN EXECUTION
# ============================================================
if __name__ == "__main__":
    print("Starting extraction with link following...")
    print("="*60)
    print(f"PostgreSQL Target: {DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")
    print("="*60)
    
    # Ensure we have the necessary libraries
    try:
        import psycopg2
    except ImportError:
        print("psycopg2 is not installed. Please install it with: pip install psycopg2-binary")
        exit(1)
    
    # Extract data
    films_data = extract_films_data_with_follow_links()
    
    if films_data:
        # Save to PostgreSQL and JSON
        save_to_postgresql_and_json(films_data)
        
        # Run example queries
        query_database_examples()
    else:
        print("No films were extracted!")

Starting extraction with link following...
PostgreSQL Target: localhost:5432/films_db

Processing 1. Avatar (2009)...
  Box Office: $2,923,710,708
  URL: https://en.wikipedia.org/wiki/Avatar_(2009_film)
  Director: James Cameron
  Country: 20th Century Fox   Lightstorm Entertainment   Dune Entertainment   Ingenious Film Partners  

Processing 2. Avengers: Endgame (2019)...
  Box Office: $2,797,501,328
  URL: https://en.wikipedia.org/wiki/Avengers:_Endgame
  Director: Anthony Russo Joe Russo
  Country: United States

Processing 3. Avatar: The Way of Water (2022)...
  Box Office: $2,334,484,620
  URL: https://en.wikipedia.org/wiki/Avatar:_The_Way_of_Water
  Director: Unknown
  Country: Various

Processing 4. Titanic (1997)...
  Box Office: $2,257,906,828
  URL: https://en.wikipedia.org/wiki/Titanic_(1997_film)
  Director: James Cameron
  Country: United States

Processing 5. Ne Zha 2 (2025)...
  Box Office: $2,215,690,000
  URL: https://en.wikipedia.org/wiki/Ne_Zha_2
  Director: Jiaozi
 